In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *


print('Importing Helper Funcions...')
# Add the directory containing Constants to the system path
if ('./HelperFunctions' not in sys.path):
    sys.path.append('./HelperFunctions')  # project root
# Import the constants
import HelperFunctions

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

print('Importing the values of the constants...')
# Add the directory containing Constants to the system path
if ('./Constants' not in sys.path):
    sys.path.append('./Constants')  # project root
# Import the constants
import CTE

print('Importing function to load the DataFrames...')
# Add the directory containing LoadDataFrames to the system path
if ('./DataFrameUtils' not in sys.path):
    sys.path.append('./DataFrameUtils')  # project root
from Files import filename_to_dataframe
from DFCleaning import *

print('Importing Functions to make cut masks...')
# Add the directory containing Constants to the system path
if ('./CutMasks' not in sys.path):
    sys.path.append('./CutMasks')  # project root
# Import the constants
import CutMasks

print('Importing Helper Funcions...')
# Add the directory containing Constants to the system path
if ('./HelperFunctions' not in sys.path):
    sys.path.append('./HelperFunctions')  # project root
# Import the constants
import HelperFunctions

print('Importing make_df...')
# Add the directory containing Constants to the system path
if ('./makedf' not in sys.path):
    sys.path.append('./makedf')  # project root
# Import the constants
import make_cc1pidf

# Load DataFrames

In [ ]:
## Check keys in each file
test_file = "/scratch/7DayLifetime/lpelegrina/cc1pi_wiremod_YZ_test.df"
print("keys in test_file")
splh.print_keys(test_file)

## Check split multiplicity
print("mc_bnb_cosmic_file n_split: %d" %splh.get_n_split(test_file))

In [ ]:
## Define keys to load
print('MC dataframes')
n_max_concat = 2 ## for big files, each key could have more than one split
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
test_df = splh.load_dfs(test_file, keys2load, n_max_concat)
print('test data loaded!')

# Filter DataFrame

In [ ]:
#Perform duplication validation
print("duplication for Spring Production BNB + Cosmic sample")
find_duplicate_run_evt_combinations(test_df['hdr'])

In [ ]:
plot_duplicate_run_subrun_evt_distribution(test_df["hdr"], "test_df")

In [ ]:
### Filter the hdr DataFrame first, then filter other DataFrames by matching with the hdr DataFrame
test_df["hdr"] = filter_unique_events(test_df["hdr"])
find_duplicate_run_evt_combinations(test_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    test_df[key] = filter_using_hdr(test_df[key], test_df["hdr"])

# Check normalization

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect the offbeam data fudge factor and scale for offbeam data
#n_record_spill_data = get_n_evt(data_bnb_light_dfs['hdr'])
#n_gates_data = len(data_bnb_light_dfs["pot"])

#n_record_spill_offbeam_data = get_n_evt(data_offbeam_light_dfs['hdr'])
#n_gates_offbeam_data = data_offbeam_light_dfs["hdr"][data_offbeam_light_dfs["hdr"]['first_in_subrun'] == 1]['noffbeambnb'].sum()

#p_trig_data = n_record_spill_data / n_gates_data
#p_trig_offbeam_data = n_record_spill_offbeam_data / n_gates_offbeam_data

#f_factor = (p_trig_data - p_trig_offbeam_data) / (1 - p_trig_offbeam_data)
#print("f_factor: %f" %f_factor)

#intime_gate_scale = (1. - f_factor) * (n_gates_data + 0.) / (n_gates_offbeam_data + 0.)
#print("intime_gate_scale: %f" %intime_gate_scale)

In [ ]:
## Collect pot scale for MC
mc_tot_pot = test_df["hdr"]['pot'].sum()
#mc_low_th_tot_pot = mc_rockbox_th1to100_dfs["hdr"]['pot'].sum()

data_tot_pot = 3.63462e+18
#data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
#data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
#data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))
#print("mc_low_thtot_pot: %e" %(mc_low_th_tot_pot))

#print("data_tot_pot: %e" %(data_tot_pot))
#print("data_tot_TOR860: %e" %(data_tot_TOR860))
#print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
#mc_low_th_scale = target_pot / mc_low_th_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))
#print("MC Low Th. POT scale: %.3f" %(mc_low_th_scale))

In [ ]:
## Comparison between observed and expected total number of recorded spills
n_evt_mc = get_n_evt(test_df["hdr"])
#n_evt_mc_low_th = get_n_evt(mc_rockbox_th1to100_dfs["hdr"])

#print("n_evt_data_onbeam: %d" %n_record_spill_data)
#print("n_evt_exp.: %f" %(n_evt_mc * mc_pot_scale + n_evt_mc_low_th * mc_low_th_scale +n_record_spill_offbeam_data * intime_gate_scale))
print("- n_evt_mc: %f" %(n_evt_mc * mc_pot_scale))
#print("- n_evt_mc_low_th: %f" %(n_evt_mc_low_th * mc_low_th_scale))
#print("- n_evt_data_offbeam: %f" %(n_record_spill_offbeam_data * intime_gate_scale))

# Do truth matching

In [ ]:
evt_df = test_df['cc1pi']
hdr_df = test_df['hdr']
nu_df = test_df['nudf']

In [ ]:
new_columns = []
for c in nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

In [ ]:
matchdf = ph.multicol_merge(evt_df.reset_index(), nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
matchdf = matchdf.set_index(evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
matchdf = matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)

In [ ]:
evt_df = matchdf

# Test background composition

In [ ]:
slc_df = evt_df
HelperFunctions.print_purity(slc_df)

In [ ]:
obvious_cosmic_mask = slc_df.slc.cut.obvious_cosmic == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask])

In [ ]:
t0_mask = slc_df.slc.cut.t0
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask])

In [ ]:
is_inside_FV_mask = slc_df.slc.cut.inside_FV == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask])

In [ ]:
nu_score_mask = slc_df.slc.cut.nu_score == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask])

In [ ]:
track_mask = slc_df.slc.cut.track == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask])

In [ ]:
shower_mask = slc_df.slc.cut.shower == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask])

In [ ]:
chi2_mask = slc_df.slc.cut.MIP_candidates == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask])

In [ ]:
angle_mask = slc_df.slc.cut.angle == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask & angle_mask])

In [ ]:
proton_BDT_mask = slc_df.slc.cut.proton_BDT == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask & angle_mask & proton_BDT_mask])

In [ ]:
containment_mask = slc_df.slc.cut.containment == True
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask & angle_mask & containment_mask])

In [ ]:
filtered_df = slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask & angle_mask & containment_mask]

In [ ]:
michel_mask = slc_df.slc.cut.michel == True
HelperFunctions.print_purity(filtered_df[michel_mask])

In [ ]:
extra_pion_mask = slc_df.slc.cut.extra_pion == True
HelperFunctions.print_purity(filtered_df[michel_mask & extra_pion_mask])

In [ ]:
proton_BDT_mask = slc_df.slc.cut.proton_BDT == True
#HelperFunctions.print_purity(filtered_df[michel_mask & extra_pion_mask &  proton_BDT_mask])
HelperFunctions.print_purity(filtered_df[proton_BDT_mask])

In [ ]:
def set_hep_style():
    mpl.rcParams.update({

        # -------------------------------------------------
        # Figure
        # -------------------------------------------------
        'figure.figsize': (8, 6),
        'figure.dpi': 100,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',

        # -------------------------------------------------
        # Fonts
        # -------------------------------------------------
        'font.family': 'sans-serif',
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 16,
        'legend.fontsize': 12,

        # -------------------------------------------------
        # Axes
        # -------------------------------------------------
        'axes.linewidth': 2.0,
        'axes.edgecolor': 'black',
        'axes.grid': False,

        # -------------------------------------------------
        # Ticks (ROOT-like)
        # -------------------------------------------------
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.major.size': 8,
        'ytick.major.size': 8,
        'xtick.major.width': 2,
        'ytick.major.width': 2,
        'xtick.minor.size': 4,
        'ytick.minor.size': 4,
        'xtick.minor.width': 1.5,
        'ytick.minor.width': 1.5,
        'xtick.top': True,
        'ytick.right': True,

        # -------------------------------------------------
        # Histograms
        # -------------------------------------------------
        'hist.bins': 50,

        # -------------------------------------------------
        # Lines
        # -------------------------------------------------
        'lines.linewidth': 2.5,
        'lines.markersize': 6,

        # -------------------------------------------------
        # Legend
        # -------------------------------------------------
        'legend.frameon': True,
        'legend.framealpha': 1.0,
        'legend.edgecolor': 'black',
        'legend.fancybox': False,

    })

In [ ]:
# Color map
MC_COLORS = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # olive
    "#17becf",  # cyan

    "#393b79",  # dark blue
    "#637939",  # dark green
    "#8c6d31",  # dark olive
    "#843c39",  # dark red
    "#7b4173",  # dark purple
    "#3182bd",  # steel blue
    "#31a354",  # teal green
    "#756bb1",  # muted violet
    "#636363",  # dark gray
    "#e6550d",  # burnt orange
]

In [ ]:
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba

# --------------------------------------------------
# Config class (columns, bins, colors, labels)
# --------------------------------------------------
@dataclass
class HistogramConfig:
    data_column: tuple               # Column with the data to plot
    bins: np.ndarray = np.linspace(-0.5, 0.5, 51)
    color_map: dict = field(default_factory=dict)
    xlabel: str = 'Score'
    ylabel: str = 'Entries'
    title: str = None

# --------------------------------------------------
# Plotting function
# --------------------------------------------------
def plot_stacked_histogram(
    df,
    config: HistogramConfig,
    type_column: tuple,
    first_per_slice: bool = False,
):
    """
    Plot a stacked histogram using a HistogramConfig and an external type column.

    Parameters
    ----------
    df : pandas DataFrame
        The dataframe containing the data.
    config : HistogramConfig
        Configuration object with data_column, bins, colors, labels, and title.
    type_column : tuple
        Column used for stacking.
    first_per_slice : bool, optional
        If True, plot only the first PFP per slice
        (grouped by ['__ntuple', 'entry', 'rec.slc..index']).
        Default is False (use all rows).
    """

    slice_levels = ['__ntuple', 'entry', 'rec.slc..index']

    # Optionally reduce to first PFP per slice
    if first_per_slice:
        df = (
            df
            .groupby(level=slice_levels, sort=False)
            .first()
        )

    # Extract series
    data = df[config.data_column]
    types = df[type_column]

    # Unique type values (preserve order of appearance)
    type_values = list(types.dropna().unique())

    # Build stack data
    stack_data = {
        t: data[types == t].dropna()
        for t in type_values
    }

    # Colors (list-based, stable)
    colors = [MC_COLORS[i % len(MC_COLORS)] for i in range(len(type_values))]

    # Create figure
    fig, ax = plt.subplots()

    # Filled stack
    ax.hist(
        [stack_data[t] for t in type_values],
        bins=config.bins,
        stacked=True,
        histtype='stepfilled',
        color=colors,
        alpha=0.1,
        linewidth=0,
    )

    # Outline stack
    ax.hist(
        [stack_data[t] for t in type_values],
        bins=config.bins,
        stacked=True,
        histtype='step',
        color=colors,
        linewidth=2.0,
    )

    # Legend
    legend_handles = [
        Patch(
            facecolor=to_rgba(c, 0.1),
            edgecolor=to_rgba(c, 1.0),
            linewidth=2.0,
            label=t
        )
        for c, t in zip(colors, type_values)
    ]

    ax.set_xlabel(config.xlabel)
    ax.set_ylabel(config.ylabel)
    if config.title:
        ax.set_title(config.title)

    ax.legend(handles=legend_handles, loc='upper right')

    plt.show()


In [ ]:
MIP_df = filtered_df[michel_mask & extra_pion_mask & ~CutMasks.exiting_pfp_mask(filtered_df) & CutMasks.is_MIP_candidate_mask(filtered_df)]
# Define config
config_bdt_proton = HistogramConfig(
    data_column=('pfp','trk','bdt_proton_score','','',''),
    bins=np.linspace(-0.5, 0.5, 51),
    xlabel='BDT proton score',
    ylabel='Entries',
    title='BDTProtonScore'
)

config_bdt_muon_pion = HistogramConfig(
    data_column=('pfp','trk','bdt_muon_pion_score','','',''),
    bins=np.linspace(-0.5, 0.5, 51),
    xlabel='BDT muon/pion score',
    ylabel='Entries',
    title='BDTMuonPionScore'
)

config_chi2mu = HistogramConfig(
    data_column=('pfp','trk','chi2pid','best','chi2_muon',''),
    bins=np.linspace(0, 60, 41),
    xlabel=r'$\text{primary tracks } \chi^{2}_{\mu}$',
    ylabel='Entries',
    title='Chi2mu'
)
config_chi2p = HistogramConfig(
    data_column=('pfp','trk','chi2pid','best','chi2_proton',''),
    bins=np.linspace(0, 300, 41),
    xlabel=r'$\text{primary tracks } \chi^{2}_{p}$',
    ylabel='Entries',
    title='Chi2p'
)

config_chi2exppol0 = HistogramConfig(
    data_column=('pfp','trk','chi2_exp_pol_3var','','',''),
    bins=np.linspace(0, 1.025, 42),
    xlabel=r'$\frac{\chi^{2}/ndf_{\mathrm{pol0}}}{\chi^{2}/ndf_{\mathrm{exp}}}$',
    ylabel='Entries',
    title='Chi2expol'
)

config_frac50 = HistogramConfig(
    data_column=('pfp','trk','frac50','','',''),
    bins=np.linspace(0, 1, 41),
    xlabel='residual range frac 50%',
    ylabel='Entries',
    title='Frac50'
)

config_max_daughter_hits = HistogramConfig(
    data_column=('pfp','max_daughter_hits','','','',''),
    bins=np.linspace(1, 200, 51),
    xlabel='max daughter hits',
    ylabel='Entries',
    title='max daughter hits'
)

config_scatter_angle = HistogramConfig(
    data_column=('pfp','scatter_angle_ratio','','','',''),
    bins=np.linspace(0, 1, 51),
    xlabel='scatter angle ratio',
    ylabel='Entries',
    title='scatter angle ratio'
)

configs = [ config_bdt_proton, config_bdt_muon_pion, 
            config_chi2mu, config_chi2p, config_chi2exppol0,
            config_frac50, config_max_daughter_hits, config_scatter_angle]

# Now we pass type_column explicitly
plot_stacked_histogram(
    MIP_df,
    config=config_frac50,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)


In [ ]:
filtered_df[('pfp','trk','bdt_proton_score','','','')]

In [ ]:
candidate_df = filtered_df[michel_mask & extra_pion_mask & CutMasks.is_MIP_candidate_mask(filtered_df)]

In [ ]:
'''
def get_mu_pi_vars(group):
    # group has 2 rows (2 PFPs)

    exiting = CutMasks.exiting_pfp_mask(group)

    # Case 1️⃣: exactly one exiting PFP
    if exiting.sum() == 1:
        p_mu = group.loc[exiting].pfp.trk.mcsP.fwdP_muon.iloc[0]
        p_pi = group.loc[~exiting].pfp.trk.rangeP.p_pion.iloc[0]
        cos_theta_mu =  group.loc[exiting].pfp.trk.dir.z.iloc[0]
        cos_theta_pi =  group.loc[~exiting].pfp.trk.dir.z.iloc[0]
        muon_contained = False

    # Case 2️⃣: no exiting PFP
    else:
        group_sorted = group.sort_values(
            ('pfp','trk','bdt_muon_pion_score','','','')
        )

        p_mu = group_sorted.pfp.trk.rangeP.p_muon.iloc[1]
        p_pi = group_sorted.pfp.trk.rangeP.p_pion.iloc[0]
        cos_theta_mu =  group_sorted.pfp.trk.dir.z.iloc[1]
        cos_theta_pi =  group_sorted.pfp.trk.dir.z.iloc[0]
        muon_contained = True

    return pd.Series({
        'reco_p_mu': p_mu,
        'reco_p_pi': p_pi,
        'cos_theta_mu': cos_theta_mu,
        'cos_theta_pi': cos_theta_pi,
        'muon_contained': muon_contained,
    })
'''

In [ ]:
group_levels = ['__ntuple','entry', 'rec.slc..index']

In [ ]:
'''
resolved = (
    candidate_df
    .groupby(level=group_levels)
    .apply(get_mu_pi_vars)
)
'''

In [ ]:
'''
p_mu_series = resolved['reco_p_mu']
p_pi_series = resolved['reco_p_pi']
cos_theta_mu_series = resolved['cos_theta_mu']
cos_theta_pi_series = resolved['cos_theta_pi']
muon_contained_series = resolved['muon_contained']
'''

In [ ]:
'''
slcdf = pd.DataFrame({
    'muon_contained': muon_contained_series,
    'reco_p_mu': p_mu_series,
    'reco_p_pi': p_pi_series,
    'reco_cos_theta_mu_1': cos_theta_mu_series,
    'reco_cos_theta_pi_1': cos_theta_pi_series
})
    # Add column to df
slcdf.columns = pd.MultiIndex.from_tuples([('slc','measure_var',col,'','','') for col in slcdf.columns])        
candidate_df = candidate_df.join(slcdf)
'''

In [ ]:
# Define config
config_p_mu = HistogramConfig(
    data_column=('slc','measure_var','reco_p_mu','','',''),
    bins=np.linspace(0, 3, 41),
    xlabel='Pmu',
    ylabel='Entries',
    title='Pmu'
)

config_TLE_p_pi = HistogramConfig(
    data_column=('slc','measure_var','TLE_p_pi','','',''),
    bins=np.linspace(0.1,0.83, 41),
    xlabel='Ppi_TLE',
    ylabel='Entries',
    title='Ppi'
)

config_range_p_pi = HistogramConfig(
    data_column=('slc','measure_var','range_p_pi','','',''),
    bins=np.linspace(0.1, 0.8, 41),
    xlabel='range_p_pi',
    ylabel='Entries',
    title='Ppi'
)

config_costheta_mu = HistogramConfig(
    data_column=('slc','measure_var','reco_cos_theta_mu','','',''),
    bins=np.linspace(0, 1, 41),
    xlabel='costhetamu',
    ylabel='Entries',
    title='costhetamu'
)

config_costheta_pi = HistogramConfig(
    data_column=('slc','measure_var','reco_cos_theta_pi','','',''),
    bins=np.linspace(0, 1, 41),
    xlabel='costhetapi',
    ylabel='Entries',
    title='costhetapi'
)

# Now we pass type_column explicitly
plot_stacked_histogram(
    #candidate_df[candidate_df.slc.measure_var.muon_contained],
    candidate_df,
    config=config_range_p_pi,
    type_column=('truth','nu_categ','','','',''),
    first_per_slice = True
)


In [ ]:
'''
import os
import ROOT
import subprocess
from ROOT import std

# 1. Setup Paths
base_path = "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod"
# Let's get the ROOT lib directory dynamically to ensure it's correct
try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Load the "Big" ROOT dependency blocks
# Loading these covers almost all physics class requirements (Hist, Geom, Graf, etc.)
root_libs = ["libRint","libROOTGpadv7","libMathMore","libCore", "libRIO", "libNet", "libHist", "libGraf", "libGraf3d", "libGpad", "libTree", "libMathCore", "libThread", "libMatrix", "libGeom","libROOTHist"]
for lib in root_libs:
    ROOT.gSystem.Load(lib)

def load_custom_class(class_name, extension="cpp"):
    source_file = os.path.join(base_path, f"{class_name}.{extension}")
    so_file = os.path.join(base_path, f"{class_name}_{extension}.so")
    
    # Try to load existing
    if os.path.exists(so_file):
        # We check return status: 0 = loaded, 1 = already loaded, -1 = failure
        if ROOT.gSystem.Load(so_file) >= 0:
            print(f"📦 Loaded: {class_name}")
            return True
    
    # If load failed or file missing, compile
    print(f"🛠️  Compiling: {class_name}...")
    return ROOT.gSystem.CompileMacro(source_file, "kO") >= 0

# 4. Run the sequence
if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        from ROOT import Hypfit
        h_fit = Hypfit()
        print("🚀 Success! All libraries and dependencies are active.")
'''

In [ ]:
candidate_df.pfp.trk.truth.p.columns

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the columns and categories
# Let's compare Hypfit to the standard Range momentum
col_a = ('slc','measure_var','TLE_p_pi','','','') # Hypfit result
col_b = ('truth', 'true_var', 'true_p_pi', '','','') # Reference/Standard result
type_column=('truth','nu_categ','','','','')

data = (candidate_df[col_a] - candidate_df[col_b])/candidate_df[col_a]
types = candidate_df[type_column]

# Unique type values (preserve order of appearance)
type_values = list(types.dropna().unique())

# Build stack data
stack_data = {
    t: data[types == t].dropna()
    for t in type_values
}

# Colors (list-based, stable)
colors = [MC_COLORS[i % len(MC_COLORS)] for i in range(len(type_values))]

# Create figure
fig, ax = plt.subplots()

# Filled stack
ax.hist(
    [stack_data[t] for t in type_values],
    bins=np.linspace(-1.5, 1.5, 41),
    stacked=True,
    histtype='stepfilled',
    color=colors,
    alpha=0.1,
    linewidth=0,
)

# Outline stack
ax.hist(
    [stack_data[t] for t in type_values],
    bins=np.linspace(-1.5, 1.5, 41),
    stacked=True,
    histtype='step',
    color=colors,
    linewidth=2.0,
)

# Legend
legend_handles = [
    Patch(
        facecolor=to_rgba(c, 0.1),
        edgecolor=to_rgba(c, 1.0),
        linewidth=2.0,
        label=t
    )
    for c, t in zip(colors, type_values)
]

ax.set_xlabel("res")
ax.set_ylabel("evts")

ax.legend(handles=legend_handles, loc='upper right')

plt.show()


In [ ]:
#best_hit_df = best_hit_df.sort_values('rr', ascending=True)

In [ ]:
hit0_df.index

In [ ]:
filtered_df[filtered_df.pfp.trk.truth.p.parent != 10000000].pfp.trk.truth.p.parent

In [ ]:
for column in slc_df.columns:
    print(column)